In [ ]:
%%configure -f
{
    "defaultLakehouse": { "name": "SilverLakehouse" }
}

# Silver - Course Reviews (AI)  ·  centerpiece
Reads the **raw course reviews** (free text in ADLS, shortcutted into `BronzeLakehouse/Files/data`)
and uses **Fabric AI functions** to re-derive the review signal from the unstructured
`studentSurveyResponse` text - instead of trusting the pre-baked `CourseReviewSignal` table.
**AI functions used** (built-in LLM, no API key):
- `ai.analyze_sentiment` -> sentiment label
- `ai.summarize`         -> one-line summary
- `ai.classify`          -> would-recommend (recommend / not_recommend)
- `ai.extract`           -> typed numeric sentiment score + boolean topic themes
**Silver output**: `nick.course_review_signal_ai`, then reproduces **demo question 4**
(does AI-derived sentiment track the grade earned?).
> **Prerequisite:** the tenant's *Copilot / AI functions* switch must be enabled on the
> capacity, and the workspace must run on Runtime 1.3+. Without it the `ai.*` calls error.

In [ ]:
import requests
import pandas as pd
from pyspark.sql import functions as F

try:
    import notebookutils
    WORKSPACE_ID = notebookutils.runtime.context["currentWorkspaceId"]
except Exception:
    WORKSPACE_ID = spark.conf.get("trident.workspace.id")

ONELAKE = "onelake.dfs.fabric.microsoft.com"
SILVER_SCHEMA = "cr"

def resolve_item_id(display_name: str, item_type: str) -> str:
    tok = notebookutils.credentials.getToken("pbi")
    url = f"https://api.fabric.microsoft.com/v1/workspaces/{WORKSPACE_ID}/items?type={item_type}"
    r = requests.get(url, headers={"Authorization": f"Bearer {tok}"}, timeout=60)
    r.raise_for_status()
    for it in r.json()["value"]:
        if it["displayName"] == display_name:
            return it["id"]
    raise ValueError(f"{item_type} '{display_name}' not found in workspace {WORKSPACE_ID}")

def item_path(item_id: str, *segments: str) -> str:
    base = f"abfss://{WORKSPACE_ID}@{ONELAKE}/{item_id}"
    return base + ("/" + "/".join(segments) if segments else "")

BRONZE_ID = resolve_item_id("BronzeLakehouse", "Lakehouse")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")
print(f"Workspace : {WORKSPACE_ID}")
print(f"Bronze    : {BRONZE_ID}")

In [ ]:
# Read raw reviews (parquet preferred, JSON fallback) via OneLake, then take to pandas for AI.
def read_reviews():
    p_parquet = item_path(BRONZE_ID, "Files", "data", "course_reviews.parquet")
    p_json    = item_path(BRONZE_ID, "Files", "data", "course_reviews.json")
    try:
        return spark.read.parquet(p_parquet)
    except Exception:
        return spark.read.option("multiline", "true").json(p_json)

sdf = read_reviews()
print(f"Raw reviews rows: {sdf.count()}")
sdf.printSchema()

pdf = sdf.toPandas()
# free text the AI functions run on
text = pdf["studentSurveyResponse"].astype(str)

In [ ]:
# --- Fabric AI functions over the free-text responses ---
import synapse.ml.aifunc as aifunc

pdf["sentiment_label"] = text.ai.analyze_sentiment()
pdf["summary_text"]    = text.ai.summarize()
pdf["recommend_label"] = text.ai.classify("recommend", "not_recommend")

display(pdf[["courseNumber", "letterGrade", "sentiment_label", "recommend_label", "summary_text"]].head(10))

In [ ]:
# Typed extraction: a numeric sentiment score in [-1, 1] plus boolean topic themes.
extract_labels = [
    aifunc.ExtractLabel(label="sentiment_score",
        description="overall sentiment of the review as a number from -1.0 (very negative) to 1.0 (very positive).",
        type="number", max_items=1),
    aifunc.ExtractLabel(label="theme_pacing",
        description="true if the review comments on the pace/speed of the course.",
        type="boolean", max_items=1),
    aifunc.ExtractLabel(label="theme_workload",
        description="true if the review comments on the amount of work, homework, or assignments.",
        type="boolean", max_items=1),
    aifunc.ExtractLabel(label="theme_instructor",
        description="true if the review comments on the instructor/professor or teaching quality.",
        type="boolean", max_items=1),
    aifunc.ExtractLabel(label="theme_materials",
        description="true if the review comments on course materials, content, lectures, or resources.",
        type="boolean", max_items=1),
]
extracted = text.ai.extract(*extract_labels)
for col in ["sentiment_score", "theme_pacing", "theme_workload", "theme_instructor", "theme_materials"]:
    pdf[col] = extracted[col].values

display(pdf[["courseNumber", "letterGrade", "sentiment_label", "sentiment_score",
             "recommend_label", "theme_instructor", "summary_text"]].head(10))

In [ ]:
# --- Assemble the Silver signal table and coerce the AI outputs to clean types ---
def as_scalar(v):
    return v[0] if isinstance(v, list) else v

def as_bool(v):
    v = as_scalar(v)
    if isinstance(v, str):
        return v.strip().lower() in ("true", "yes", "1")
    return bool(v) if v is not None else False

def as_float(v, default=0.0):
    v = as_scalar(v)
    try:
        return float(v)
    except (TypeError, ValueError):
        return default

out = pd.DataFrame({
    "student_id":       pd.to_numeric(pdf["studentId"], errors="coerce").astype("Int64"),
    "course_id":        pd.to_numeric(pdf["courseId"], errors="coerce").astype("Int64"),
    "course_number":    pdf["courseNumber"].astype(str).str.strip(),
    "course_name":      pdf["courseName"].astype(str).str.strip(),
    "letter_grade":     pdf["letterGrade"].astype(str).str.strip(),
    "quality_points":   pd.to_numeric(pdf["qualityPoints"], errors="coerce"),
    "review_date":      pd.to_datetime(pdf["reviewDate"], errors="coerce").dt.date,
    "sentiment_label":  pdf["sentiment_label"].apply(lambda v: str(as_scalar(v)).strip().lower()),
    "sentiment_score":  pdf["sentiment_score"].apply(as_float).clip(-1.0, 1.0),
    "theme_pacing":     pdf["theme_pacing"].apply(as_bool),
    "theme_workload":   pdf["theme_workload"].apply(as_bool),
    "theme_instructor": pdf["theme_instructor"].apply(as_bool),
    "theme_materials":  pdf["theme_materials"].apply(as_bool),
    "would_recommend":  pdf["recommend_label"].apply(lambda v: str(as_scalar(v)).strip().lower() == "recommend"),
    "summary_text":     pdf["summary_text"].apply(lambda v: str(as_scalar(v)).strip()),
})
out.insert(0, "review_id", range(1, len(out) + 1))

signal_sdf = spark.createDataFrame(out)
(signal_sdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
           .saveAsTable(f"{SILVER_SCHEMA}.course_review_signal_ai"))
print(f"  nick.course_review_signal_ai    rows={spark.table(f'{SILVER_SCHEMA}.course_review_signal_ai').count()}")

In [ ]:
# --- Demo question 4: does AI-derived sentiment track the grade earned? ---
signal = spark.table(f"{SILVER_SCHEMA}.course_review_signal_ai")
grade_band = (F.when(F.col("quality_points") >= 3.0, "A/B (>=3.0)")
               .when(F.col("quality_points") >= 2.0, "C (2.0-2.9)")
               .otherwise("D/F (<2.0)"))

display(
    signal.withColumn("grade_band", grade_band)
          .groupBy("grade_band")
          .agg(F.count("*").alias("reviews"),
               F.round(F.avg("sentiment_score"), 2).alias("avg_ai_sentiment"),
               F.round(F.avg(F.col("would_recommend").cast("double")), 2).alias("pct_recommend"))
          .orderBy(F.col("avg_ai_sentiment").desc())
)